# Phase 9: Feature Engineering & Selection (Housing)

**Objective:** We reached the algorithmic ceiling in Week 5. Today, we test 5 custom feature hypotheses in isolation against our `Baseline_Linear` champion model. Then, we use VIF, L1 Regularization (Lasso), and RFE to mathematically select the optimal feature space and eliminate noise.

In [12]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent.parent))

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.feature_selection import RFE

from src.utils.transformers import HousingFeatureEngineer
from src.utils.stats import calculate_vif
from src.models.evaluator import evaluate_regression, save_training_report
from src.models.registry import save_model, load_latest_model
from src.visualization.plots import plot_actual_vs_predicted, set_journalism_style

import warnings
warnings.filterwarnings('ignore') # Clean notebook output
from sklearn.preprocessing import QuantileTransformer
from src.models.evaluator import save_training_report
from src.models.registry import save_model

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from scipy.stats import loguniform

set_journalism_style()
SUBFOLDER = "housing_week_6"

# 1. Load Data
df = pd.read_csv(Path.cwd().parent.parent / "datasets" / "processed" / "housing_cleaned.csv")
X = df.drop(columns=['price'])
y = df['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Verify Week 5 Champion is loaded as Benchmark
champion = load_latest_model("housing")
y_pred_bench = champion.predict(X_test)
print("--- WEEK 5 BENCHMARK ---")
evaluate_regression(y_test, y_pred_bench)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
--- WEEK 5 BENCHMARK ---
--- Regression Evaluation ---
MAE:  960,123.17
MSE:  1,728,299,877,704.57
RMSE: 1,314,648.20
R²:   0.6581
-----------------------------
Insight: The model explains 65.8% of the variance in property prices. Predictions deviate by an average of $960,123.17 absolute error.


### 1. Hypothesis Testing in Isolation
We use our custom `HousingFeatureEngineer` to test 5 hypotheses individually. If an engineered feature lowers the RMSE, it passes Phase 1.

In [13]:
def test_hypothesis(flag_name: str, numeric_adds: list = [], categorical_adds: list = []):
    """Helper to dynamically build and test a pipeline."""
    num_feat = ['area', 'bedrooms', 'bathrooms', 'stories', 'parking'] + numeric_adds
    cat_feat = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea', 'furnishingstatus'] + categorical_adds
    
    prep = ColumnTransformer([
        ('num', RobustScaler(), num_feat),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_feat)
    ])
    
    kwargs = {flag_name: True}
    pipe = Pipeline([
        ('engineer', HousingFeatureEngineer(**kwargs)),
        ('prep', prep),
        ('model', TransformedTargetRegressor(LinearRegression(), func=np.log1p, inverse_func=np.expm1))
    ])
    
    pipe.fit(X_train, y_train)
    metrics, _ = evaluate_regression(y_test, pipe.predict(X_test), return_dict=True)
    return metrics['rmse']

print("--- HYPOTHESIS TESTING ---")
rmse_h1 = test_hypothesis('h1_density', numeric_adds=['area_per_room'])
rmse_h2 = test_hypothesis('h2_amenity', numeric_adds=['luxury_score'])
rmse_h3 = test_hypothesis('h3_poly', numeric_adds=['area_squared'])
rmse_h4 = test_hypothesis('h4_estate', categorical_adds=['is_estate'])
rmse_h5 = test_hypothesis('h5_rooms', numeric_adds=['total_rooms'])

--- HYPOTHESIS TESTING ---
--- Regression Evaluation ---
MAE:  961,430.40
MSE:  1,723,066,313,103.88
RMSE: 1,312,656.21
R²:   0.6591
-----------------------------
Insight: The model explains 65.9% of the variance in property prices. Predictions deviate by an average of $961,430.40 absolute error.
--- Regression Evaluation ---
MAE:  960,123.17
MSE:  1,728,299,877,704.57
RMSE: 1,314,648.20
R²:   0.6581
-----------------------------
Insight: The model explains 65.8% of the variance in property prices. Predictions deviate by an average of $960,123.17 absolute error.
--- Regression Evaluation ---
MAE:  978,021.26
MSE:  1,737,638,132,751.64
RMSE: 1,318,195.03
R²:   0.6562
-----------------------------
Insight: The model explains 65.6% of the variance in property prices. Predictions deviate by an average of $978,021.26 absolute error.
--- Regression Evaluation ---
MAE:  941,290.86
MSE:  1,671,679,034,004.88
RMSE: 1,292,934.27
R²:   0.6693
-----------------------------
Insight: The model expla

### 2. Feature Selection: VIF & L1 Regularization
*(Note for User: Review the RMSE outputs above. If H5 Total Rooms worked, we must drop bedrooms/bathrooms to prevent collinearity. We test this using VIF.)*

Next, we run Lasso (L1). Lasso shrinks useless features to exactly 0.0.

In [14]:
# 1. Apply ALL passing hypotheses (Assume all passed for this code block setup)
engineer = HousingFeatureEngineer(h1_density=True, h2_amenity=True, h3_poly=True, h4_estate=True, h5_rooms=True)
X_train_eng = engineer.transform(X_train)

# 2. VIF Check
print("--- VIF ANALYSIS ---")
display(calculate_vif(X_train_eng))

# 3. L1 Selection
num_all = ['area', 'bedrooms', 'bathrooms', 'stories', 'parking', 'area_per_room', 'luxury_score', 'area_squared', 'total_rooms']
cat_all = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea', 'furnishingstatus', 'is_estate']

prep = ColumnTransformer([('num', RobustScaler(), num_all), ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_all)])
X_train_prep = prep.fit_transform(X_train_eng)
features = prep.get_feature_names_out()

# We fit a Lasso regressor on the log-transformed target
lasso = Lasso(alpha=0.01, random_state=42)
lasso.fit(X_train_prep, np.log1p(y_train))

l1_coefs = pd.DataFrame({'Feature': features, 'Weight': lasso.coef_})
l1_coefs['Abs_Weight'] = l1_coefs['Weight'].abs()
print("\n--- L1 LASSO SELECTION (Zero-Weight Features Dropped) ---")
display(l1_coefs[l1_coefs['Abs_Weight'] == 0])

--- VIF ANALYSIS ---


,Feature,VIF
0,bedrooms,inf
1,total_rooms,inf
2,bathrooms,inf
3,area,252.022339
4,area_per_room,60.096385
5,area_squared,27.237521
6,stories,6.933663
7,luxury_score,2.135404
8,parking,1.933486



--- L1 LASSO SELECTION (Zero-Weight Features Dropped) ---


,Feature,Weight,Abs_Weight
1,num__bedrooms,0.0,0.0
5,num__area_per_room,0.0,0.0
7,num__area_squared,0.0,0.0
12,cat__hotwaterheating_yes,-0.0,0.0
13,cat__airconditioning_yes,0.0,0.0
14,cat__prefarea_yes,0.0,0.0
15,cat__furnishingstatus_semi-furnished,0.0,0.0
17,cat__is_estate_1,-0.0,0.0


In [15]:
# 1. Apply ALL passing hypotheses (Assume all passed for this code block setup)
engineer = HousingFeatureEngineer(h1_density=True, h2_amenity=True, h3_poly=True, h4_estate=True, h5_rooms=True)
X_train_eng = engineer.transform(X_train)

# 2. VIF Check
print("--- VIF ANALYSIS ---")
display(calculate_vif(X_train_eng))

# 3. L1 Selection
num_all = ['area', 'bedrooms', 'bathrooms', 'stories', 'parking', 'area_per_room', 'luxury_score', 'area_squared', 'total_rooms']
cat_all = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea', 'furnishingstatus', 'is_estate']

prep = ColumnTransformer([('num', RobustScaler(), num_all), ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_all)])
X_train_prep = prep.fit_transform(X_train_eng)
features = prep.get_feature_names_out()

# We fit a Lasso regressor on the log-transformed target
lasso = Lasso(alpha=0.01, random_state=42)
lasso.fit(X_train_prep, np.log1p(y_train))

l1_coefs = pd.DataFrame({'Feature': features, 'Weight': lasso.coef_})
l1_coefs['Abs_Weight'] = l1_coefs['Weight'].abs()
print("\n--- L1 LASSO SELECTION (Zero-Weight Features Dropped) ---")
display(l1_coefs[l1_coefs['Abs_Weight'] == 0])

--- VIF ANALYSIS ---


,Feature,VIF
0,bedrooms,inf
1,total_rooms,inf
2,bathrooms,inf
3,area,252.022339
4,area_per_room,60.096385
5,area_squared,27.237521
6,stories,6.933663
7,luxury_score,2.135404
8,parking,1.933486



--- L1 LASSO SELECTION (Zero-Weight Features Dropped) ---


,Feature,Weight,Abs_Weight
1,num__bedrooms,0.0,0.0
5,num__area_per_room,0.0,0.0
7,num__area_squared,0.0,0.0
12,cat__hotwaterheating_yes,-0.0,0.0
13,cat__airconditioning_yes,0.0,0.0
14,cat__prefarea_yes,0.0,0.0
15,cat__furnishingstatus_semi-furnished,0.0,0.0
17,cat__is_estate_1,-0.0,0.0


### 3. Assembling the Final Week 6 Champion (Housing)

**Domain Knowledge vs. Algorithmic Selection:**
While L1 Lasso Regularization provided a mathematical baseline by zeroing out features like `bedrooms`, relying blindly on it results in a loss of human context. A buyer mentally anchors on "3 bedrooms" differently than total area. Therefore, we are overriding the algorithmic selector to restore `bedrooms`.

Furthermore, our feature space has changed significantly. Wrapping a linear model in a new feature space requires re-tuning. To satisfy the internship requirements, we will employ both `GridSearchCV` and `RandomizedSearchCV` to find the absolute optimal hyperparameters for our engineered dataset.

In [18]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LinearRegression, HuberRegressor
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import randint, uniform
import time

# 1. Engineered Data Pipeline (Restoring 'bedrooms' against Lasso's advice)
final_engineer = HousingFeatureEngineer(h2_amenity=True, h4_estate=True)
final_num = ['area', 'bedrooms', 'bathrooms', 'stories', 'parking', 'luxury_score'] 
final_cat = ['mainroad', 'guestroom', 'basement', 'airconditioning', 'prefarea', 'furnishingstatus', 'is_estate'] 

final_prep = ColumnTransformer([
    ('num_scale', RobustScaler(), final_num),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), final_cat)
])

# 2. Define the Top 3 Models from Week 5 (to test on the new engineered data)
top_3_models = {
    "Engineered_Linear": Pipeline([('engineer', final_engineer), ('prep', final_prep), ('model', TransformedTargetRegressor(LinearRegression(), func=np.log1p, inverse_func=np.expm1))]),
    "Engineered_Huber": Pipeline([('engineer', final_engineer), ('prep', final_prep), ('model', TransformedTargetRegressor(HuberRegressor(max_iter=1000), func=np.log1p, inverse_func=np.expm1))]),
    "Engineered_RandomForest": Pipeline([('engineer', final_engineer), ('prep', final_prep), ('model', RandomForestRegressor(random_state=42))])
}

# 3. Define Grids and Distributions
param_grids = {
    "Engineered_Linear": {"model__regressor__fit_intercept": [True, False]},
    "Engineered_Huber": {"model__regressor__epsilon": [1.2, 1.35, 1.5], "model__regressor__alpha": [0.0001, 0.001]},
    "Engineered_RandomForest": {"model__max_depth": [None, 10, 20], "model__min_samples_split": [2, 5]}
}

param_dists = {
    "Engineered_Linear": {"model__regressor__fit_intercept": [True, False]},
    "Engineered_Huber": {"model__regressor__epsilon": uniform(1.1, 0.5), "model__regressor__alpha": uniform(0.0001, 0.001)},
    "Engineered_RandomForest": {"model__max_depth": randint(5, 30), "model__min_samples_split": randint(2, 10)}
}

results = []
trained_models = {}

print("--- HYPERPARAMETER TUNING ENGINEERED MODELS ---")

for name, pipeline in top_3_models.items():
    print(f"\n--- Tuning {name} ---")
    
    # A. GridSearchCV
    grid_search = GridSearchCV(estimator=pipeline, param_grid=param_grids[name], cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)
    
    # B. RandomizedSearchCV
    random_search = RandomizedSearchCV(estimator=pipeline, param_distributions=param_dists[name], n_iter=5, cv=5, scoring="neg_root_mean_squared_error", random_state=42, n_jobs=-1)
    
    # Fit both (we'll time the GridSearch for the logs, but use the best overall)
    start_time = time.time()
    grid_search.fit(X_train, y_train)
    random_search.fit(X_train, y_train)
    
    # Pick the absolute best estimator between the two searches
    if grid_search.best_score_ > random_search.best_score_:
        best_model = grid_search.best_estimator_
        best_params = grid_search.best_params_
        search_type = "GridSearch"
    else:
        best_model = random_search.best_estimator_
        best_params = random_search.best_params_
        search_type = "RandomSearch"
        
    y_pred_tuned = best_model.predict(X_test)
    metrics_tuned, _ = evaluate_regression(y_test, y_pred_tuned, return_dict=True)
    
    metrics_tuned["Model"] = f"Tuned_{name}"
    metrics_tuned["Time (s)"] = round(time.time() - start_time, 2)
    
    results.append(metrics_tuned)
    trained_models[f"Tuned_{name}"] = best_model
    
    print(f"Winner: {search_type}")
    print(f"Best Params: {best_params}")

# Display Final Comparison Table
comparison_df = pd.DataFrame(results)[["Model", "mae", "mse", "rmse", "r2", "Time (s)"]]
comparison_df.columns = ["Model", "MAE", "MSE", "RMSE", "R² Score", "Time (s)"]
display(comparison_df.sort_values(by="RMSE"))

--- HYPERPARAMETER TUNING ENGINEERED MODELS ---

--- Tuning Engineered_Linear ---
--- Regression Evaluation ---
MAE:  941,290.86
MSE:  1,671,679,034,004.88
RMSE: 1,292,934.27
R²:   0.6693
-----------------------------
Insight: The model explains 66.9% of the variance in property prices. Predictions deviate by an average of $941,290.86 absolute error.
Winner: RandomSearch
Best Params: {'model__regressor__fit_intercept': True}

--- Tuning Engineered_Huber ---
--- Regression Evaluation ---
MAE:  975,896.61
MSE:  1,748,881,888,713.12
RMSE: 1,322,452.98
R²:   0.6540
-----------------------------
Insight: The model explains 65.4% of the variance in property prices. Predictions deviate by an average of $975,896.61 absolute error.
Winner: RandomSearch
Best Params: {'model__regressor__alpha': np.float64(0.00025601864044243655), 'model__regressor__epsilon': np.float64(1.1779972601681015)}

--- Tuning Engineered_RandomForest ---
--- Regression Evaluation ---
MAE:  1,018,327.48
MSE:  1,974,480,181

,Model,MAE,MSE,RMSE,R² Score,Time (s)
0,Tuned_Engineered_Linear,9.412909e+05,1.671679e+12,1.292934e+06,0.669274,3.74
1,Tuned_Engineered_Huber,9.758966e+05,1.748882e+12,1.322453e+06,0.654000,2.46
2,Tuned_Engineered_RandomForest,1.018327e+06,1.974480e+12,1.405162e+06,0.609367,5.70


### 3. Conclusion: The Limits of Isolated Testing & Regularization
1. **Isolated Testing:** In strict isolation, Hypothesis 4 (`is_estate`) was the only feature to meaningfully drop the RMSE (from $1.314M to $1.292M). 
2. **Multicollinearity:** The VIF analysis successfully triggered `inf` (infinity) scores for `bedrooms`, `bathrooms`, and `total_rooms`. This mathematically proved perfect collinearity, meaning we *must* drop the sub-components if we use the aggregated feature.
3. **Lasso Selection:** L1 Regularization was ruthless. It zeroed out nearly all our engineered features. This teaches us a crucial lesson: isolated univariate testing doesn't account for multivariate redundancy. 
*Next Step: We will abandon isolated testing and use Incremental Forward Stacking to find the optimal combination of synergistic features.*